# Scaling

Feature scaling means bringing numerical variables with different ranges into a common scale/distribution.

Many ML algorithms are sensitive to the magnitude of features. Without scaling, a feature with a large numerical range mathematically appears "more important" than a feature with a small range, even though that has nothing to do with its actual importance.

* Sensitive (scaling is essential):
    * Distance-based algorithms: KNN, K-Means clustering
    * Models optimized via gradient descent: Linear/Logistic Regression, Neural Networks (without scaling, convergence is very slow or may not happen at all)
    * Models using regularization: Ridge, Lasso (since the penalty term depends on coefficient magnitude)

* Non-Sensitive:
    * Tree-based models: Decision Tree, Random Forest, Gradient Boosting (XGBoost, LightGBM). These split features one at a time based on thresholds, so they're unaffected by scale.

## Main Methods

### StandardScaler (Z-score)
Produces mean 0, standard deviation 1. The most commonly used, **works well when the data is roughly normal.**

### MinMaxScaler
Squeezes values into a range, typically 0-1. Preferred when a bounded range is needed (e.g. neural network input layers), but very sensitive to outliers.

### RobustScaler
Uses median and IQR instead of mean/std. Unaffected by outliers. A safe choice for skewed data with many outliers.

### MaxAbsScaler
Scales values into a -1 to 1 range without shifting the center of the data (useful for sparse data, e.g. text/TF-IDF).

### Normalizer
Unlike the others, this scales rows (each sample) rather than columns, making each row's vector norm equal to 1. Typically used in text/NLP contexts.


# Practical Rules

1. **Fit only on train**: `fit_transform` on the train set, `transform` (without fitting) on the test set. Never leak test set statistics into the scaler.

2. **Watch out for outliers**: If there are extreme outliers, StandardScaler/MinMaxScaler can be misleading, so either handle outliers first or use RobustScaler.

3. **Do it inside a Pipeline**: If using cross-validation, put scaling inside a `Pipeline` object; doing it manually risks leakage in each fold.


>When you scale manually before splitting into folds, the scaler's mean/std (or min/max) get computed from both the train and validation portions of each fold combined. This means the model indirectly learns statistical information about the validation fold before ever being evaluated on it. That's leakage. <br>
> **With Pipeline + cross_val_score (or GridSearchCV), the mechanism works like this:**<br>
> 1. For each fold, sklearn automatically calls scaler.fit() using only that fold's train portion <br>
> 2. It then applies transform() only (not fit) to the validation portion, using those same learned parameters<br>
> 3. When moving to the next fold, this process repeats from scratch, with a different train/validation split

4. **Don't scale categorical/ordinal columns**: One-hot encoded columns are already 0/1, no need to scale. Ordinal encoded columns are sometimes scaled and sometimes not, depending on the model.






# Skewness

Skewness is a statistic showing how far a distribution deviates from symmetry. In pandas, the .skew() method computes this directly for each column.

| Skewness value | Interpretation |
|---|---|
| -0.5 to +0.5 | Roughly symmetric, no transform needed |
| ±0.5 to ±1 | Moderately skewed, transform may be considered |
| Beyond ±1 | Highly skewed, transform usually needed |
| Positive value | Right-skewed, tail on the right |
| Negative value | Left-skewed, tail on the left |

## Skewness Visualation

- **Histogram / KDE plot**: lets you see the shape of the distribution visually, helping distinguish whether it's a single outlier or a genuine skew
- **Q-Q plot (Quantile-Quantile plot)**: shows how well the data fits a normal distribution. If points lie on a straight line, it's close to normal; if they curve, there's skewness. This gives a more reliable visual confirmation than the skew number alone.
Why use both: sometimes a single extreme outlier can artificially inflate the skewness value, when in fact removing that outlier reveals a reasonably shaped distribution. The histogram/Q-Q plot lets you tell the difference.


## Decision flow (step by step)

### Automatic Decision
Instead of manually checking skew and deciding per column, `PowerTransformer` already finds the optimal lambda automatically. In practice there are two approaches:


- **Manual/controlled approach**: compute skew, select columns above the threshold, decide yourself which transform (log/Box-Cox/Yeo-Johnson) to apply
- **Automated approach**: apply `PowerTransformer(method='yeo-johnson')` to all numeric columns and let the algorithm find its own optimal lambda for each column



# Log/Power transform

For skewed distributions, log-transform or `PowerTransformer` (Box-Cox, Yeo-Johnson) can be applied before or instead of scaling. These reshape the distribution **toward normal**, not just change the scale.


# Log Transform
**Logic:** Apply `log(x)` to each value (usually natural log, `np.log`).

**What it does:** Compresses right-skewed distributions. It shrinks the gap between large values while relatively preserving the gap between small values. For example, in a column like `SalePrice`, the difference between 500,000 and 600,000 becomes much more "reasonable" in scale after taking the log.

#### Managing x=0:
**Constraint:** Doesn't work for `x <= 0` (log is undefined there). This is why `log1p(x)` = `log(1+x)` is typically used instead, **since it's defined even when `x=0`.**

#### Reversal is needed:
**Reversing it:** If the model was trained on log-scale, predictions need to be converted back to the original scale using `expm1` (the inverse of `log1p`).

# Power Transform
Power transform is applying a power (exponent) function to data in order to change the shape of its distribution

Log transform can actually be thought of as a special case of power transform.

λ = 0.5 → square root

λ = 0 → log transform (special case)

λ = -1 → 1/x (reciprocal)

λ = 2, 3... → squaring, cubing

# Box-Cox and Yeo-Johnson:
Box-Cox and Yeo-Johnson, mentioned earlier, are **automatically optimized versions (finding the best λ)** of this general power transform family:

$$
x^{(\lambda)} = \begin{cases} \dfrac{x^\lambda - 1}{\lambda} & \lambda \neq 0 \\ \ln(x) & \lambda = 0 \end{cases}
$$

Box-Cox only works on positive values (`x > 0`).

**Yeo-Johnson:** An extended version of Box-Cox, with the formula made a bit more complex so it also works on negative and zero values, but the logic is the same: find the best lambda and transform the data according to that power.

### Why does λ = 0 give the log transform?

Note the formula is `(x^λ - 1) / λ`, not just `x^λ`. Plugging in λ = 0 directly gives a **0/0** indeterminate form (`x^0 - 1 = 1 - 1 = 0`, and the denominator `λ = 0` too), so it's undefined at that point.

The log transform comes from taking the **limit as λ → 0** (via L'Hôpital's rule or a Taylor expansion), which works out to exactly `ln(x)`:

$$\lim_{\lambda \to 0} \frac{x^\lambda - 1}{\lambda} = \ln(x)$$

So λ = 0 is defined as a special case in the formula precisely to avoid this undefined point: as λ approaches zero, the formula's behavior converges to the log function, so `ln(x)` is used directly at λ = 0.

## How is the "best" lambda found?

`PowerTransformer` (or `scipy.stats.boxcox`) tries different lambda values on the data and automatically picks the one that produces a distribution **closest to normal** (via maximum likelihood estimation). No need to manually try "should I take the square root or the log": the algorithm optimizes this for you.